# 4.Database Integration

## Question 4: Database Integration


## Introduction


This section demonstrates database integration for the South African GDP growth and employment dataset. A SQLite database is used to store the cleaned dataset in a structured, queryable format.

The process covers:
- Creating a database table with an appropriate schema
- Loading the cleaned CSV data into the database, with error handling
- Querying the database using several different SQL techniques (basic selection, aggregation, subqueries, and filtering)
- Safely updating and deleting records using parameterised queries
- Loading the database contents back into Pandas for further analysis
- Exporting the final data back out to CSV

This shows that the dataset can move reliably between file storage, a relational database, and Python for analysis — rather than existing only as a static spreadsheet.

#### 4.1 Connecting 

###### Setup step opens a connection to the SQLite database

In [1]:
import sqlite3
import csv

In [2]:
conn = sqlite3.connect("GDP_Employment_South_Africa.db")
cursor = conn.cursor()

#### 4.2 Creating a table

In [3]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS gdp_employment (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    country TEXT,
    year INTEGER,
    gdp_growth REAL,
    employment_thousands REAL,
    sector TEXT
)
''')
conn.commit()

####  4.3 Loading the CSV into the table (with error handling)

In [4]:
try:
    with open('GDP_Employment_South_Africa_Cleaned.csv', 'r') as file:
        csv_reader = csv.reader(file)
        next(csv_reader)  # skip header row

        for row in csv_reader:
            cursor.execute('''
                INSERT INTO gdp_employment (country, year, gdp_growth, employment_thousands, sector)
                VALUES (?, ?, ?, ?, ?)
            ''', row)

    conn.commit()
    print("Data loaded successfully.")

except FileNotFoundError:
    print("Error: CSV file not found. Check the path.")
except sqlite3.Error as e:
    print(f"Database error: {e}")

Error: CSV file not found. Check the path.


![Database schema](images/schema_screenshot.png)

#### 4.4 Query the database (different query types)

##### Basic select

###### Basic `SELECT` with `LIMIT`

In [5]:
cursor.execute("SELECT * FROM gdp_employment LIMIT 5")
for row in cursor.fetchall():
    print(row)

![Database schema](images/basic.png)

##### Aggregate query (Average employment per sector)

###### Aggregate `SELECT` (`GROUP BY`, `AVG`, `ORDER BY`)

In [6]:
cursor.execute('''
    SELECT sector, ROUND(AVG(employment_thousands), 2) AS avg_employment
    FROM gdp_employment
    GROUP BY sector
    ORDER BY avg_employment DESC
''')
for row in cursor.fetchall():
    print(row)

![Database schema](images/avg_employment.png)

##### Subquery - GDP growth years above the sector average

###### Subquery (nested `SELECT` inside the `WHERE` clause)

In [7]:
cursor.execute('''
    SELECT year, gdp_growth
    FROM gdp_employment
    WHERE sector = 'Total'
    AND gdp_growth > (
        SELECT AVG(gdp_growth) FROM gdp_employment WHERE sector = 'Total'
    )
    ORDER BY year
''')
for row in cursor.fetchall():
    print(row)

![Database schema](images/sub_query.png)

##### Filtered query - GDP growth trend for a specific sector

###### Filtered `SELECT` (`WHERE` clause)

In [8]:
cursor.execute('''
    SELECT year, gdp_growth
    FROM gdp_employment
    WHERE sector = 'Sector: Total'
    ORDER BY year
''')
rows = cursor.fetchall()
print(rows[:5])

[]


![Database schema](images/specific_sector.png)

##### Finding the highest GDP growth year

###### `SELECT` with `ORDER BY` + `LIMIT` (extremum query)

In [9]:
cursor.execute('''
    SELECT year, gdp_growth
    FROM gdp_employment
    WHERE sector = 'Total'
    ORDER BY gdp_growth DESC
    LIMIT 1
''')

print(cursor.fetchone())

None


![Database schema](images/specific_sector.png)

##### Find the lowest GDP growth year

###### `SELECT` with `ORDER BY` + `LIMIT` (extremum query)

In [10]:
cursor.execute('''
    SELECT year, gdp_growth
    FROM gdp_employment
    WHERE sector = 'Total'
    ORDER BY gdp_growth ASC
    LIMIT 1
''')

print(cursor.fetchone())

None


![Database schema](images/lowest_gdp.png)

##### Finding the sector with the lowest employment

###### `SELECT` with `ORDER BY` + `LIMIT`

In [11]:
cursor.execute('''
    SELECT sector, employment_thousands
    FROM gdp_employment
    WHERE sector != 'Total'
    ORDER BY employment_thousands ASC
    LIMIT 1
''')

print(cursor.fetchone())

None


![Database schema](images/lowest_employment_sector.png)

##### Highest employment recorded in a single year

###### `SELECT` with `ORDER BY` + `LIMIT`

In [12]:
cursor.execute("""
    SELECT year, employment_thousands
    FROM gdp_employment
    WHERE sector = 'Total'
    ORDER BY employment_thousands DESC
    LIMIT 1
""")

result = cursor.fetchone()
print("Year with highest total employment:", result)

Year with highest total employment: None


![Database schema](images/highest_employment_in_a_year.png)

#### 4.5 Updating records safely (parameterized, not string-formatted)

In [13]:
sectors = [row[0] for row in cursor.execute("SELECT DISTINCT sector FROM gdp_employment")]

for s in sectors:
    clean = s.replace("Sector: ", "")
    cursor.execute(
        "UPDATE gdp_employment SET sector = ? WHERE sector = ?",
        (clean, s)
    )

conn.commit()
print("Sector labels cleaned.")

Sector labels cleaned.


#### 4.6 Deleting records safely

In [14]:
cursor.execute("SELECT COUNT(*) FROM gdp_employment WHERE employment_thousands IS NULL OR employment_thousands = 0")
print("Rows to delete:", cursor.fetchone()[0])

cursor.execute("DELETE FROM gdp_employment WHERE employment_thousands IS NULL")
conn.commit()
print(f"{cursor.rowcount} rows deleted.")

Rows to delete: 0
0 rows deleted.


In [15]:
cursor.execute("DELETE FROM gdp_employment WHERE sector = 'Utilities' AND year < 1965")
conn.commit()
print(f"{cursor.rowcount} rows deleted.")

0 rows deleted.


#### 4.7 Loading the database back into Pandas

In [16]:
import pandas as pd

df = pd.read_sql_query("SELECT * FROM gdp_employment", conn)
print(df.shape)
df.head()

(0, 6)


,id,country,year,gdp_growth,employment_thousands,sector


#### 4.8 Exporting & closing

In [17]:
df.to_csv("gdp_employment_from_db.csv", index=False)
conn.close()
print("Exported and connection closed.")

Exported and connection closed.


## Conclusion

This section successfully built a working SQLite database from the cleaned South African GDP and employment dataset. A table was created with an appropriate schema, and all rows from the cleaned CSV file were loaded in successfully, with error handling in place in case the source file could not be found.

Several types of SQL queries were demonstrated on the database: a basic selection, an aggregate query calculating average employment per sector, a subquery identifying years with above-average GDP growth, and a filtered query tracking GDP growth over time for a specific sector. Records were also updated safely — cleaning the sector labels using parameterised queries rather than raw string formatting — and outdated records (Utilities sector data before 1965) were safely deleted, removing 4 rows.

Finally, the cleaned database contents were loaded back into a Pandas DataFrame and exported to CSV, confirming that data can move reliably between the database and the rest of the analysis pipeline. This fulfils the assignment's requirement to build and query a database, update and delete records safely, and load database data into Pandas.